In [ ]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

chat_generator = OllamaChatGenerator(
    model="qwen2.5:7b",
    url="http://localhost:11434",
    timeout=300,
    generation_kwargs={
        "num_predict": 3000,
        "temperature": 0.2,
    },
)

In [2]:
from dataclasses import dataclass, field
from typing import Callable
import requests

import os
import requests
from dotenv import load_dotenv

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker

load_dotenv()

# %%
from dataclasses import dataclass, field
from typing import Callable, List

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker

In [3]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

In [19]:
model_name="sentence-transformers/all-mpnet-base-v2"

from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.embedders import SentenceTransformersTextEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(
    model=model_name
)

query_embedder = SentenceTransformersTextEmbedder(
    model=model_name
)

In [17]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store=document_store)

In [20]:
@dataclass
class ToolCallingAgent:
    name: str = "ToolCallingAgent"
    llm: object = None

    instructions: str = """
You are a STRICT research agent.
...
"""

    functions: List[Callable] = field(default_factory=list)

    def __post_init__(self):
        self._system_message = ChatMessage.from_system(self.instructions)

        self.tools = [
            create_tool_from_function(fun)
            for fun in self.functions
        ]

        self._tool_invoker = ToolInvoker(
            tools=self.tools,
            raise_on_failure=False
        )

    # ✅ FIXED: properly inside class
    def run(self, messages, max_steps=3):

        messages = [self._system_message] + messages

        for _ in range(max_steps):

            response = self.llm.run(
                messages=messages,
                tools=self.tools
            )

            msg = response["replies"][0]
            messages.append(msg)

            # 🧠 no tool call → finish
            if not msg.tool_calls:
                return msg

            tool_results = self._tool_invoker.run(messages=[msg])["tool_messages"]
            messages.extend(tool_results)

        return msg

In [9]:
import requests


class TavilyWebSearch:
    def __init__(self, api_key: str, top_k: int = 3):
        self.api_key = api_key
        self.top_k = top_k

    def run(self, query: str):
        payload = {
            "api_key": self.api_key,
            "query": query,
            "max_results": self.top_k,
            "include_answer": True,
            "include_domains": [
                "arxiv.org",
                "openreview.net",
                "aclanthology.org",
                "ieee.org",
                "springer.com"
            ],
            "exclude_domains": [
                "wikipedia.org",
                "reddit.com"
            ]
        }

        try:
            resp = requests.post(
                "https://api.tavily.com/search",
                json=payload,
                timeout=15,
            )
            resp.raise_for_status()
            data = resp.json()

        except requests.RequestException as e:
            return {
                "answer": None,
                "results": [],
                "error": str(e)
            }

        results = []

        for hit in data.get("results", []):
            results.append({
                "title": hit.get("title", ""),
                "url": hit.get("url", ""),
                "content": (hit.get("content", "") or "")[:300]
            })

        return {
            "answer": data.get("answer", ""),
            "results": results
        }

In [10]:
tavily = TavilyWebSearch(
    api_key=os.environ["TAVILY_API_KEY"],
    top_k=3
)

def web_search(query: str):
    result = tavily.run(query=query)

    if isinstance(result, str):
        return {"results": [], "answer": result}

    return {
        "answer": result.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content", "")
            }
            for r in result.get("results", [])
        ]
    }

In [11]:
def download_pdf(url: str, filename: str = None) -> dict:
    try:
        headers = {"User-Agent": "Mozilla/5.0"}

        response = requests.get(url, timeout=20, headers=headers, stream=True)
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "").lower()

        # smarter PDF detection
        is_pdf = (
            "pdf" in content_type
            or url.lower().endswith(".pdf")
        )

        if not is_pdf:
            return {
                "status": "failed",
                "reason": "Not a PDF",
                "url": url
            }

        # auto filename if not given
        if filename is None:
            filename = url.split("/")[-1].split("?")[0]
            if not filename.endswith(".pdf"):
                filename += ".pdf"

        path = os.path.join("downloads", filename)
        os.makedirs("downloads", exist_ok=True)

        with open(path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)

        return {
            "status": "success",
            "file_path": path,
            "url": url
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e),
            "url": url
        }

In [13]:
import fitz  # PyMuPDF

def extract_pdf_text(file_path: str) -> str:
    doc = fitz.open(file_path)
    text = ""

    for page in doc:
        text += page.get_text()

    return text[:20000]

In [ ]:
from haystack.dataclasses import ChatMessage

SYSTEM_PROMPT = """
You extract structured scientific paper information.

Return ONLY JSON:

{
  "title": "",
  "authors": [],
  "thesis": "",
  "methodology": "",
  "contributions": [],
  "limitations": [],
  "keywords": []
}
"""

def analyze_paper(llm, text: str):
    messages = [
        ChatMessage.from_system(SYSTEM_PROMPT),
        ChatMessage.from_user(text)
    ]

    res = llm.run(messages=messages)
    return res["replies"][0].text

In [16]:
def process_paper(url: str):
    pdf = download_pdf(url)

    if pdf["status"] != "success":
        return pdf

    text = extract_pdf_text(pdf["file_path"])

    # ✅ STORE WITH EMBEDDINGS (IMPORTANT FIX)
    store_document(text, url)

    # analyze
    analysis = analyze_paper(chat_generator, text)

    return {
        "url": url,
        "analysis": analysis
    }

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

In [15]:
from haystack.dataclasses import Document

def store_document(text: str, url: str):
    doc = Document(content=text, meta={"url": url})

    result = doc_embedder.run(documents=[doc])

    document_store.write_documents(result["documents"])

In [18]:
def retrieve_papers(query: str):
    results = retriever.run(query=query, top_k=5)

    return {
        "results": [
            {
                "content": d.content[:500],
                "url": d.meta.get("url")
            }
            for d in results["documents"]
        ]
    }

In [14]:
agent = ToolCallingAgent(
    llm=chat_generator,
    functions=[web_search, process_paper, retrieve_papers, add]
)

NameError: name 'chat_generator' is not defined

In [ ]:
messages = [
    ChatMessage.from_user("What is the paper about agentic AI?")
]

In [ ]:
response = agent.run(messages)

In [93]:
for msg in response:
    print(msg.text)

Here are the key papers related to agentic AI:

1. **Paper Title**: Agentic AI Security: Threats, Defenses, Evaluation, and Open ...
   - **Link**: [https://arxiv.org/html/2510.23883v1](https://arxiv.org/html/2510.23883v1)
   - **Summary**: This paper highlights the potential of agentic AI in accelerating scientific progress and expanding the knowledge frontier.
   - **Relevance**: It addresses security aspects, which are crucial for the responsible development and deployment of agentic AI systems.

2. **Paper Title**: Agentic AI: A Comprehensive Survey of Architectures, Applications, and Future Directions
   - **Link**: [https://arxiv.org/abs/2510.25445](https://arxiv.org/abs/2510.25445)
   - **Summary**: This comprehensive survey covers the architectures, applications, and future directions of agentic AI.
   - **Relevance**: It provides a broad overview of the field, making it essential for understanding the current state and potential developments in agentic AI.

3. **Paper Title**: